# Data preparation

Downloads Financial PhraseBank (Malo et al., 2014) and exports the `Sentences_AllAgree` subset (2,264 sentences).

The 150 base sentences in `data/base_sentences.csv` were then selected **manually** from this file by human annotation (objective sentences only). That selection step is not part of the code.

In [ ]:
import io
import os
import zipfile

import pandas as pd
import requests

OUTPUTS_DIR = "../outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

URL = "https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip"

In [ ]:
def parse_file(z, filename):
    with z.open(filename) as f:
        content = f.read().decode("latin-1")

    sentences, labels = [], []
    for line in content.strip().split("\n"):
        if "@" in line:
            parts = line.rsplit("@", 1)
            if len(parts) == 2:
                sentences.append(parts[0].strip())
                labels.append(parts[1].strip())

    return pd.DataFrame({"sentence": sentences, "label": labels})


response = requests.get(URL)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    df = parse_file(z, "FinancialPhraseBank-v1.0/Sentences_AllAgree.txt")

output_path = os.path.join(OUTPUTS_DIR, "financial_phrasebank_allagree.csv")
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} sentences to: {output_path}")
print(df["label"].value_counts())